# CS2 EXP-4 — NeoBERT-250M + LoRA


## 1. Dependencies


In [1]:
import sys
import subprocess

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "--extra-index-url", "https://download.pytorch.org/whl/cu121",
    "torch==2.5.1",
    "torchvision",
    "torchaudio",
    "transformers<4.49.0",  # Avoids the CVE check enforcing PyTorch 2.6
    "peft<0.14.0",
    "accelerate",
    "numpy<2.1.0",
    "pandas",
    "scikit-learn",
    "matplotlib",
    "pyarrow",
    "joblib",
    "tqdm",
    "psutil",
    "einops",   # NeoBERT (chandar-lab/NeoBERT) remote modeling code dependency
], check=True)

# EXP-4 uses NeoBERT (chandar-lab/NeoBERT), which is shipped as `trust_remote_code`
# model code rather than a native `transformers` architecture. Its reference
# SwiGLU/attention implementation depends on `xformers`. xformers wheels are
# built against a SPECIFIC torch build, so we pin it explicitly to the release
# built for torch==2.5.1 (confirmed via its wheel metadata) rather than
# installing unpinned -- an unpinned `-U xformers` will pull the latest release,
# which requires a newer torch and will silently upgrade torch out from under
# the 2.5.1+cu121 pin above, which is what caused the earlier
# "xFormers was built for PyTorch 2.10.0+cu128" mismatch / flash-attention
# schema ImportError. We also use --no-deps so pip can't touch torch again to
# "satisfy" xformers. We force `use_unpadding=False` in case_study_2/models.py
# (see PDD sec. 5.2), so flash-attention is NOT required.
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "--no-deps",
    "xformers==0.0.28.post3",
], check=True)

print("Dependencies installed successfully for CUDA 12.1 driver!")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pyreft 0.1.0 requires evaluate>=0.4.1, which is not installed.
pyreft 0.1.0 requires gcsfs>=2024.2.0, which is not installed.
pyreft 0.1.0 requires jupyter, which is not installed.
pyreft 0.1.0 requires wandb, which is not installed.
pyreft 0.1.0 requires ydata-profiling>=4.7.0, which is not installed.
pyreft 0.1.0 requires seaborn==0.12.2, but you have seaborn 0.13.2 which is incompatible.
pyreft 0.1.0 requires transformers==4.45.1, but you have transformers 4.48.3 which is incompatible.


Dependencies installed successfully for CUDA 12.1 driver!


## 1.5 Settings


In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/EnomisLP/DiverseVul--IS-Project.git"
REPO_BRANCH = "Recovery"

WORKSPACE_ROOT = Path.cwd()
REPO_ROOT = WORKSPACE_ROOT / "DiverseVul--IS-Project"
PROJECT_DIR = REPO_ROOT / "vuln-detection"
SRC_DIR = PROJECT_DIR / "src"

DATA_ROOT = WORKSPACE_ROOT / "IntelligentSystemProject" / "VulnerabilityDetectionData"
PROCESSED_DIR = DATA_ROOT / "processed"
MANIFEST_ROOT = DATA_ROOT / "manifests"
OUTPUT_ROOT = DATA_ROOT / "outputs"

DOWNSAMPLED_PARQUET = PROCESSED_DIR / "rdiversevul_cs1_normalized_plus_abstracted_v2_downsampled20k.parquet"
MANIFEST_PATH = MANIFEST_ROOT / "cs1_shared_rotating_5fold_v1" / "project_grouped_5fold_manifest.parquet"

CODE_COLUMN = "normalized_code"
#CODE_COLUMN = "abstracted_code_v1"
CODE_COLUMN_TAG = "abstracted" if CODE_COLUMN == "abstracted_code_v1" else "normalized"

EXP4_OUTPUT_DIR = OUTPUT_ROOT / "case_study_2" / f"exp4_neobert_lora_v1_{CODE_COLUMN_TAG}"
EXP4_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

HF_CACHE_DIR = WORKSPACE_ROOT / "IntelligentSystemProject" / "hf_cache"

HF_TOKEN_VALUE = ""
if HF_TOKEN_VALUE:
    os.environ["HF_TOKEN"] = HF_TOKEN_VALUE
else:
    os.environ.pop("HF_TOKEN", None)

RANK = 16
EPOCHS = 10

TRAIN_BATCH_SIZE = 32
GRAD_ACCUM_STEPS = 2
EVAL_BATCH_SIZE = 64
NUM_WORKERS = 8

STORAGE_CAP_GB = 60

RUN_SMOKE_TEST = True
RUN_OFFICIAL = True

print("Settings loaded.")
print(f"Workspace: {WORKSPACE_ROOT}")
print(f"Repository: {REPO_ROOT}")
print(f"Data root: {DATA_ROOT}")
print(f"Downsampled parquet: {DOWNSAMPLED_PARQUET}")
print(f"Manifest: {MANIFEST_PATH}")
print(f"Hugging Face cache: {HF_CACHE_DIR}")
# (Device is detected and printed in the "Verify GPU, RAM, and storage budget"
# cell below -- DEVICE doesn't exist yet at this point in the notebook.)


## 2. Clone the repository


In [3]:
import urllib.request
import zipfile
from pathlib import Path

if not REPO_ROOT.exists():
    print(f"Downloading repository (branch: {REPO_BRANCH}) without git...")
    
    clean_url = REPO_URL.removesuffix(".git")
    zip_url = f"{clean_url}/archive/refs/heads/{REPO_BRANCH}.zip"
    zip_path = Path.cwd() / "repo_temp.zip"

   
    urllib.request.urlretrieve(zip_url, zip_path)

  
    print("Extracting files...")
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(Path.cwd())

  
    repo_name = clean_url.split("/")[-1]
    extracted_folder = Path.cwd() / f"{repo_name}-{REPO_BRANCH}"
    if extracted_folder.exists():
        extracted_folder.rename(REPO_ROOT)

 
    zip_path.unlink()
    print("Repository setup complete!")
else:
    print(f"Repository already exists at {REPO_ROOT}")


Repository already exists at /workspace/DiverseVul--IS-Project


## 3. Verify GPU, RAM, and storage budget


In [4]:
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import torch
assert torch.cuda.is_available(), "CUDA is not available!"
assert torch.cuda.device_count() == 1, f"Expected 1 GPU, but PyTorch sees {torch.cuda.device_count()}"
DEVICE = "cuda:0"
print(f"Locked to single GPU: {torch.cuda.get_device_name(0)}")
print(f"Total Visible GPUs in PyTorch: {torch.cuda.device_count()}")

Locked to single GPU: NVIDIA A100-SXM4-80GB
Total Visible GPUs in PyTorch: 1


## 4. Data availability check


In [ ]:
required_data_files = {
    "downsampled parquet": DOWNSAMPLED_PARQUET,
    "shared 5-fold manifest": MANIFEST_PATH,
}

missing = {name: path for name, path in required_data_files.items() if not path.is_file()}

if missing:
    print("Missing required data files:")
    for name, path in missing.items():
        print(f"  - {name}: {path}")
    print(
        "\nThese files are produced by notebooks/scope2_preprocessing.ipynb (downsampling + "
        "manifest-generation sections) and were previously synced through Google Drive. Copy "
        f"them into the paths above, or re-run that notebook. Keep an eye on the {STORAGE_CAP_GB} GB storage cap."
    )
    raise FileNotFoundError("Required processed data/manifest are missing; see instructions above.")
else:
    for name, path in required_data_files.items():
        size_mb = path.stat().st_size / 1e6
        print(f"Found {name}: {path} ({size_mb:.1f} MB)")


## 5. Write case_study_2 source files


In [6]:
(SRC_DIR / "case_study_2/__init__.py").parent.mkdir(parents=True, exist_ok=True)
(SRC_DIR / "case_study_2/__init__.py").write_text('')
print("Wrote", "case_study_2/__init__.py")


Wrote case_study_2/__init__.py


In [7]:
(SRC_DIR / "case_study_2/models.py").parent.mkdir(parents=True, exist_ok=True)
(SRC_DIR / "case_study_2/models.py").write_text('from __future__ import annotations\n\nimport os\nimport warnings\nfrom pathlib import Path\nfrom typing import Optional, Dict, Any, List\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom transformers import AutoConfig, AutoModel, AutoTokenizer\n\n\nDEFAULT_CODE_MODEL = "huggingface/CodeBERTa-small-v1"\nDEFAULT_CODE_TOKENIZER = "huggingface/CodeBERTa-small-v1"\n\n# NeoBERT-250M backbone (Chandar Research Lab); ships as trust_remote_code on the Hub.\nDEFAULT_NEOBERT_MODEL = "chandar-lab/NeoBERT"\nDEFAULT_NEOBERT_TOKENIZER = "chandar-lab/NeoBERT"\n\n# Substring match so NeoBERT forks/finetunes are still recognized.\n_NEOBERT_NAME_HINTS = ("neobert",)\n\n\ndef _is_neobert_model(model_name: str) -> bool:\n    """Check whether a model name refers to a NeoBERT-family checkpoint."""\n    name = (model_name or "").lower()\n    return any(hint in name for hint in _NEOBERT_NAME_HINTS)\n\n\ndef configure_huggingface_cache(hf_cache_dir: Optional[str] = None) -> None:\n    """Set the Hugging Face cache/env variables used across this project\'s downloads."""\n    if hf_cache_dir:\n        hf_cache_dir = str(hf_cache_dir)\n        os.environ.setdefault("HF_HOME", hf_cache_dir)\n        os.environ.setdefault("HUGGINGFACE_HUB_CACHE", str(Path(hf_cache_dir) / "hub"))\n    os.environ.setdefault("HF_HUB_DISABLE_XET", "1")\n    os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "0")\n    os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "120")\n    os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "120")\n    os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")\n\n\ndef _dtype_from_policy(dtype_policy: str, device: str) -> Optional[torch.dtype]:\n    """Resolve a torch dtype from a policy name and device."""\n    dtype_policy = (dtype_policy or "auto").lower()\n    device = str(device)\n    if dtype_policy == "float16":\n        return torch.float16 if device == "cuda" else torch.float32\n    if dtype_policy == "bfloat16":\n        return torch.bfloat16 if device == "cuda" and torch.cuda.is_bf16_supported() else torch.float32\n    if dtype_policy == "float32":\n        return torch.float32\n    if dtype_policy == "auto":\n        if device == "cuda" and torch.cuda.is_bf16_supported():\n            return torch.bfloat16\n        if device == "cuda":\n            return torch.float32\n        return torch.float32\n    raise ValueError(f"Unknown dtype_policy: {dtype_policy}")\n\n\ndef _apply_neobert_config_overrides(config: Any) -> Any:\n    """Disable NeoBERT sequence unpadding, since we pad batches instead of packing them."""\n    candidate_flags = ("use_unpadding", "unpad_inputs", "unpad", "pack_sequences")\n    matched = False\n    for flag in candidate_flags:\n        if hasattr(config, flag):\n            setattr(config, flag, False)\n            matched = True\n    if not matched:\n        warnings.warn(\n            "[models] No known unpadding flag found on the NeoBERT config "\n            f"(checked: {candidate_flags}); verify attention-mask correctness manually."\n        )\n    return config\n\n\ndef load_code_tokenizer(\n    tokenizer_name: str = DEFAULT_CODE_TOKENIZER,\n    hf_cache_dir: Optional[str] = None,\n    trust_remote_code: Optional[bool] = None,\n):\n    """Load the tokenizer for a code model, auto-detecting NeoBERT\'s trust_remote_code need."""\n    configure_huggingface_cache(hf_cache_dir)\n    if trust_remote_code is None:\n        trust_remote_code = _is_neobert_model(tokenizer_name)\n    return AutoTokenizer.from_pretrained(\n        tokenizer_name,\n        use_fast=True,\n        cache_dir=hf_cache_dir,\n        trust_remote_code=trust_remote_code,\n    )\n\n\ndef load_code_encoder(\n    model_name: str = DEFAULT_CODE_MODEL,\n    dtype_policy: str = "auto",\n    device: Optional[str] = None,\n    freeze: bool = True,\n    hf_cache_dir: Optional[str] = None,\n    trust_remote_code: Optional[bool] = None,\n) -> nn.Module:\n    """Load a code backbone encoder, applying NeoBERT-specific safeguards when needed."""\n    device = device or ("cuda" if torch.cuda.is_available() else "cpu")\n    configure_huggingface_cache(hf_cache_dir)\n    dtype = _dtype_from_policy(dtype_policy, device)\n\n    is_neobert = _is_neobert_model(model_name)\n    if trust_remote_code is None:\n        trust_remote_code = is_neobert\n\n    kwargs: Dict[str, Any] = {"cache_dir": hf_cache_dir, "trust_remote_code": trust_remote_code}\n    if dtype is not None:\n        kwargs["torch_dtype"] = dtype\n\n    if is_neobert:\n        # NeoBERT\'s fused flash/memory-efficient SDPA backends crash with a\n        # device-side CUDA assert on real (non-toy) batches on this\n        # environment; force the math (unfused) backend instead.\n        torch.backends.cuda.enable_flash_sdp(False)\n        torch.backends.cuda.enable_mem_efficient_sdp(False)\n        torch.backends.cuda.enable_math_sdp(True)\n\n        # Patch the config before the backbone is instantiated.\n        config = AutoConfig.from_pretrained(\n            model_name, cache_dir=hf_cache_dir, trust_remote_code=trust_remote_code\n        )\n        config = _apply_neobert_config_overrides(config)\n        kwargs["config"] = config\n\n    model = AutoModel.from_pretrained(model_name, **kwargs)\n    model.to(device)\n\n    if freeze:\n        for param in model.parameters():\n            param.requires_grad = False\n        model.eval()\n\n    return model\n\n\ndef mean_pool_last_hidden(last_hidden_state: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:\n    """Mean-pool token embeddings over non-padded positions."""\n    mask = attention_mask.unsqueeze(-1).to(last_hidden_state.dtype)\n    summed = (last_hidden_state * mask).sum(dim=1)\n    denom = mask.sum(dim=1).clamp(min=1.0)\n    return summed / denom\n\n\ndef cls_pool_last_hidden(last_hidden_state: torch.Tensor) -> torch.Tensor:\n    """Take the [CLS]-position embedding."""\n    return last_hidden_state[:, 0, :]\n\n\nclass CodeSequenceClassifier(nn.Module):\n    """Backbone encoder + linear classification head over pooled embeddings."""\n\n    def __init__(\n        self,\n        model_name: str = DEFAULT_CODE_MODEL,\n        num_labels: int = 1,\n        freeze_backbone: bool = False,\n        pooling: str = "mean",\n        dtype_policy: str = "auto",\n        hf_cache_dir: Optional[str] = None,\n        trust_remote_code: Optional[bool] = None,\n        enforce_fp32_head: Optional[bool] = None,\n    ) -> None:\n        """Build the backbone and classification head."""\n        super().__init__()\n        device = "cuda" if torch.cuda.is_available() else "cpu"\n        self.backbone = load_code_encoder(\n            model_name=model_name,\n            dtype_policy=dtype_policy,\n            device=device,\n            freeze=freeze_backbone,\n            hf_cache_dir=hf_cache_dir,\n            trust_remote_code=trust_remote_code,\n        )\n        hidden_size = int(self.backbone.config.hidden_size)\n        self.classification_head = nn.Linear(hidden_size, num_labels)\n        self.pooling = pooling\n\n        # Pooling and the classification head run in float32 regardless of\n        # ambient autocast dtype, to avoid baking a bf16/fp16 NaN/Inf from\n        # NeoBERT\'s attention stack into the trainable head.\n        if enforce_fp32_head is None:\n            enforce_fp32_head = _is_neobert_model(model_name)\n        self.enforce_fp32_head = enforce_fp32_head\n\n    @property\n    def config(self):\n        """Expose the underlying backbone config to peft."""\n        return self.backbone.config\n\n    @property\n    def device(self) -> torch.device:\n        """Expose the device where parameters reside."""\n        return next(self.parameters()).device\n\n    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor, **kwargs: Any) -> torch.Tensor:\n        """Encode, pool, and classify a batch, returning logits."""\n        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask, **kwargs)\n        hidden = outputs.last_hidden_state\n\n        if self.enforce_fp32_head:\n            hidden = hidden.float()\n            attention_mask_for_pool = attention_mask.float()\n        else:\n            attention_mask_for_pool = attention_mask\n\n        if self.pooling == "cls":\n            pooled = cls_pool_last_hidden(hidden)\n        else:\n            pooled = mean_pool_last_hidden(hidden, attention_mask_for_pool)\n\n        if self.enforce_fp32_head:\n            # Disable autocast so the head matmul isn\'t downcast back to bf16/fp16.\n            with torch.autocast(device_type=pooled.device.type, enabled=False):\n                logits = self.classification_head(pooled.float())\n        else:\n            logits = self.classification_head(pooled)\n\n        if logits.ndim > 1 and logits.size(-1) == 1:\n            return logits.squeeze(-1)\n        return logits\n\n\ndef count_trainable_parameters(model: nn.Module) -> Dict[str, int]:\n    """Report trainable vs. total parameter counts."""\n    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)\n    total = sum(p.numel() for p in model.parameters())\n    return {\n        "trainable_parameters": int(trainable),\n        "total_parameters": int(total),\n        "trainable_percent": float(100.0 * trainable / max(total, 1)),\n    }\n\n\ndef infer_lora_target_modules(model: nn.Module) -> List[str]:\n    """Guess which attention projection module names LoRA should target."""\n    module_names = [name for name, _ in model.named_modules()]\n    candidate_sets = [\n        ["qkv"],\n        ["q_proj", "v_proj"],\n        ["query", "value"],\n        ["in_proj"],\n    ]\n    for candidates in candidate_sets:\n        if all(any(name.endswith(candidate) or f".{candidate}" in name for name in module_names) for candidate in candidates):\n            return candidates\n    return ["query", "value"]\n\n\ndef create_lora_sequence_classifier(\n    model_name: str = DEFAULT_CODE_MODEL,\n    rank: int = 8,\n    lora_alpha: int = 16,\n    lora_dropout: float = 0.05,\n    pooling: str = "mean",\n    dtype_policy: str = "auto",\n    hf_cache_dir: Optional[str] = None,\n    trust_remote_code: Optional[bool] = None,\n):\n    """Wrap a CodeSequenceClassifier with a LoRA adapter via peft."""\n    from peft import LoraConfig, get_peft_model\n\n    base = CodeSequenceClassifier(\n        model_name=model_name,\n        freeze_backbone=False,\n        pooling=pooling,\n        dtype_policy=dtype_policy,\n        hf_cache_dir=hf_cache_dir,\n        trust_remote_code=trust_remote_code,\n    )\n    target_modules = infer_lora_target_modules(base)\n    config = LoraConfig(\n        r=rank,\n        lora_alpha=lora_alpha,\n        target_modules=target_modules,\n        lora_dropout=lora_dropout,\n        bias="none",\n        task_type="FEATURE_EXTRACTION",\n        modules_to_save=["classification_head"],\n    )\n    return get_peft_model(base, config)\n\n\ndef get_lora_model(\n    model_name: str = DEFAULT_CODE_MODEL,\n    rank: int = 8,\n    lora_alpha: int = 16,\n    pooling: str = "mean",\n    trust_remote_code: Optional[bool] = None,\n):\n    """Build a LoRA-adapted sequence classifier for the given backbone."""\n    return create_lora_sequence_classifier(\n        model_name=model_name,\n        rank=rank,\n        lora_alpha=lora_alpha,\n        pooling=pooling,\n        trust_remote_code=trust_remote_code,\n    )\n\n')
print("Wrote", "case_study_2/models.py")


Wrote case_study_2/models.py


In [8]:
(SRC_DIR / "case_study_2/data_loader.py").parent.mkdir(parents=True, exist_ok=True)
(SRC_DIR / "case_study_2/data_loader.py").write_text('from __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom typing import Optional, Dict, Any, List\n\nimport pandas as pd\nimport torch\nfrom torch.utils.data import Dataset, DataLoader\n\n\nEMPTY_CODE_SENTINEL = "EMPTY_CODE_SAMPLE"\n\n\nclass CodeTextDataset(Dataset):\n    """PyTorch Dataset yielding raw code text plus label/id/project for one row."""\n\n    def __init__(\n        self,\n        dataframe: pd.DataFrame,\n        code_column: str = "normalized_code",\n        label_column: str = "label",\n        source_id_column: str = "source_row_id",\n        project_column: str = "project",\n    ) -> None:\n        """Copy the frame and replace empty code with a sentinel token."""\n        self.df = dataframe.copy().reset_index(drop=True)\n        self.code_column = code_column\n        self.label_column = label_column\n        self.source_id_column = source_id_column\n        self.project_column = project_column\n\n        self.df[self.code_column] = self.df[self.code_column].fillna("").astype(str)\n        empty_mask = self.df[self.code_column].str.strip().eq("")\n        if empty_mask.any():\n            self.df.loc[empty_mask, self.code_column] = EMPTY_CODE_SENTINEL\n\n    def __len__(self) -> int:\n        """Return the number of rows."""\n        return int(len(self.df))\n\n    def __getitem__(self, idx: int) -> Dict[str, Any]:\n        """Return one row as a plain dict."""\n        row = self.df.iloc[idx]\n        return {\n            "code": str(row[self.code_column]),\n            "label": int(row[self.label_column]),\n            "source_row_id": int(row[self.source_id_column]),\n            "project": str(row[self.project_column]),\n        }\n\n\n@dataclass\nclass TransformerBatchCollator:\n    """Tokenize a batch of raw-code dicts into padded model inputs."""\n\n    tokenizer: Any\n    max_length: int = 512\n    pad_to_multiple_of: Optional[int] = 8\n\n    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:\n        """Tokenize and collate a list of row dicts into one batch."""\n        texts = [feature["code"] for feature in features]\n        enc = self.tokenizer(\n            texts,\n            truncation=True,\n            max_length=self.max_length,\n            padding=True,\n            pad_to_multiple_of=self.pad_to_multiple_of,\n            return_tensors="pt",\n        )\n\n        labels = torch.tensor([feature["label"] for feature in features], dtype=torch.float32)\n        source_row_ids = torch.tensor([feature["source_row_id"] for feature in features], dtype=torch.long)\n        projects = [feature["project"] for feature in features]\n\n        enc["labels"] = labels\n        enc["label"] = labels\n        enc["source_row_id"] = source_row_ids\n        enc["project"] = projects\n        return enc\n\n\ndef create_dataloader(\n    dataframe: pd.DataFrame,\n    tokenizer: Any,\n    batch_size: int = 16,\n    max_length: int = 512,\n    shuffle: bool = False,\n    code_column: str = "normalized_code",\n    label_column: str = "label",\n    source_id_column: str = "source_row_id",\n    project_column: str = "project",\n    num_workers: int = 0,\n) -> DataLoader:\n    """Build a DataLoader that tokenizes code rows on the fly."""\n    dataset = CodeTextDataset(\n        dataframe=dataframe,\n        code_column=code_column,\n        label_column=label_column,\n        source_id_column=source_id_column,\n        project_column=project_column,\n    )\n    collator = TransformerBatchCollator(\n        tokenizer=tokenizer,\n        max_length=max_length,\n        pad_to_multiple_of=8 if torch.cuda.is_available() else None,\n    )\n    return DataLoader(\n        dataset,\n        batch_size=batch_size,\n        shuffle=shuffle,\n        drop_last=False,\n        num_workers=num_workers,\n        pin_memory=torch.cuda.is_available(),\n        collate_fn=collator,\n    )\n\n\ndef get_pos_weight(dataframe: pd.DataFrame, label_column: str = "label") -> torch.Tensor:\n    """Compute the negative/positive ratio for BCEWithLogitsLoss\'s pos_weight."""\n    y = dataframe[label_column].astype(int).values\n    neg = int((y == 0).sum())\n    pos = int((y == 1).sum())\n    if pos == 0:\n        return torch.tensor([1.0], dtype=torch.float32)\n    return torch.tensor([neg / pos], dtype=torch.float32)\n\n\ndef get_class_weights(dataframe: pd.DataFrame, label_column: str = "label") -> torch.Tensor:\n    """Alias for get_pos_weight, used as the BCE positive-class weight."""\n    return get_pos_weight(dataframe, label_column=label_column)\n\n')
print("Wrote", "case_study_2/data_loader.py")


Wrote case_study_2/data_loader.py


In [9]:
(SRC_DIR / "case_study_2/exp4/__init__.py").parent.mkdir(parents=True, exist_ok=True)
(SRC_DIR / "case_study_2/exp4/__init__.py").write_text('')
print("Wrote", "case_study_2/exp4/__init__.py")


Wrote case_study_2/exp7/__init__.py


In [ ]:
(SRC_DIR / "case_study_2/exp4/exp4_lora.py").parent.mkdir(parents=True, exist_ok=True)
(SRC_DIR / "case_study_2/exp4/exp4_lora.py").write_text('from __future__ import annotations\n\nimport gc\nimport torch\nimport torch.nn as nn\nfrom torch.optim import AdamW\nimport numpy as np\n\nfrom case_study_2.data_loader import create_dataloader, get_class_weights\nfrom case_study_2.models import get_lora_model, count_trainable_parameters, DEFAULT_NEOBERT_MODEL\nfrom case_study_2.training_utils import EarlyStoppingConfig, run_training_with_early_stopping\n\n\ndef forward_lora(model: nn.Module, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:\n    """Run the model and squeeze a trailing singleton logit dimension."""\n    logits = model(input_ids=input_ids, attention_mask=attention_mask)\n    if logits.ndim > 1 and logits.size(-1) == 1:\n        logits = logits.squeeze(-1)\n    return logits\n\n\ndef train_lora_model(\n    train_df,\n    val_df,\n    tokenizer,\n    rank,\n    lora_alpha=16,\n    learning_rate=2e-4,\n    epochs=3,\n    batch_size=16,\n    grad_accum_steps=2,\n    eval_batch_size=32,\n    num_workers=2,\n    device="cuda",\n    hf_cache_dir=None,\n    code_column="normalized_code",\n    max_length=512,\n    verbose=True,\n    log_every_steps=500,\n    log_prefix="",\n    early_stopping=True,\n    patience=2,\n    min_delta=1e-4,\n    min_epochs=2,\n):\n    """Fine-tune a LoRA-adapted NeoBERT classifier with validation-PR-AUC early stopping."""\n    train_loader = create_dataloader(\n        train_df, tokenizer, batch_size=batch_size, max_length=max_length,\n        shuffle=True, num_workers=num_workers, code_column=code_column,\n    )\n    val_loader = create_dataloader(\n        val_df, tokenizer, batch_size=eval_batch_size, max_length=max_length,\n        shuffle=False, num_workers=num_workers, code_column=code_column,\n    )\n\n    model = get_lora_model(\n        model_name=DEFAULT_NEOBERT_MODEL, rank=rank, lora_alpha=lora_alpha,\n        trust_remote_code=True,\n    ).to(device)\n\n    is_cuda = (device == "cuda") or (hasattr(device, "type") and device.type == "cuda")\n    total_steps_per_epoch = -(-len(train_df) // batch_size)\n\n    if verbose:\n        stats = count_trainable_parameters(model)\n        print(\n            f"{log_prefix}[lora] rank={rank} | train_rows={len(train_df)} | val_rows={len(val_df)} | "\n            f"batch_size={batch_size} | grad_accum={grad_accum_steps} | steps/epoch={total_steps_per_epoch} | "\n            f"trainable={stats[\'trainable_parameters\']:,} ({stats[\'trainable_percent\']:.3f}%) | "\n            f"total={stats[\'total_parameters\']:,}"\n        )\n        if is_cuda:\n            print(f"{log_prefix}[lora] VRAM after model load: {torch.cuda.memory_allocated()/1e9:.2f} GB")\n\n    pos_weight = get_class_weights(train_df).to(device)\n    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)\n    optimizer = AdamW(model.parameters(), lr=learning_rate)\n\n    es_config = EarlyStoppingConfig(\n        max_epochs=epochs, patience=patience, min_delta=min_delta,\n        min_epochs=min_epochs, enabled=early_stopping,\n    )\n    all_scores, best_epoch, history = run_training_with_early_stopping(\n        model, forward_lora, train_loader, val_loader, optimizer, criterion, device,\n        es_config=es_config, is_cuda=is_cuda, grad_accum_steps=grad_accum_steps,\n        verbose=verbose, log_every_steps=log_every_steps, log_prefix=log_prefix,\n        phase_name="lora", total_steps_per_epoch=total_steps_per_epoch,\n    )\n\n    del train_loader, val_loader, criterion, optimizer\n    if is_cuda:\n        torch.cuda.empty_cache()\n        torch.cuda.reset_peak_memory_stats()\n\n    return np.array(all_scores), model, history\n\n\ndef score_model(model, df, tokenizer, device, code_column="normalized_code",\n                 max_length=512, batch_size=32, num_workers=2):\n    """Score an arbitrary dataframe with a trained model: no gradient, no early-stopping bookkeeping."""\n    loader = create_dataloader(\n        df, tokenizer, batch_size=batch_size, max_length=max_length,\n        shuffle=False, num_workers=num_workers, code_column=code_column,\n    )\n    model.eval()\n    all_scores = []\n    with torch.no_grad():\n        for batch in loader:\n            input_ids = batch["input_ids"].to(device, non_blocking=True)\n            attention_mask = batch["attention_mask"].to(device, non_blocking=True)\n            with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):\n                logits = forward_lora(model, input_ids, attention_mask)\n                scores = torch.sigmoid(logits)\n            all_scores.extend(scores.float().cpu().numpy())\n    return np.array(all_scores)\n\n\ndef train_lora_model_safe(*args, max_retries=2, **kwargs):\n    """Call train_lora_model, halving the batch size and retrying on CUDA OOM."""\n    batch_size = kwargs.pop("batch_size", 16)\n    grad_accum_steps = kwargs.pop("grad_accum_steps", 2)\n\n    attempt = 0\n    while True:\n        try:\n            return train_lora_model(\n                *args, batch_size=batch_size, grad_accum_steps=grad_accum_steps, **kwargs\n            )\n        except torch.cuda.OutOfMemoryError:\n            attempt += 1\n            gc.collect()\n            torch.cuda.empty_cache()\n            if attempt > max_retries or batch_size <= 2:\n                raise\n            new_batch_size = max(2, batch_size // 2)\n            new_grad_accum_steps = grad_accum_steps * max(1, batch_size // new_batch_size)\n            print(\n                f"[lora] CUDA OOM at batch_size={batch_size}; retrying "\n                f"(attempt {attempt}/{max_retries}) with batch_size={new_batch_size}, "\n                f"grad_accum_steps={new_grad_accum_steps} (effective batch size unchanged)."\n            )\n            batch_size, grad_accum_steps = new_batch_size, new_grad_accum_steps\n')
print("Wrote", "case_study_2/exp4/exp4_lora.py")

In [ ]:
(SRC_DIR / "case_study_2/exp4/exp4_nested_rank.py").parent.mkdir(parents=True, exist_ok=True)
(SRC_DIR / "case_study_2/exp4/exp4_nested_rank.py").write_text('from __future__ import annotations\n\nimport gc\nimport json\nimport time\nfrom dataclasses import dataclass, asdict\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nfrom typing import Any, Dict, List, Optional\n\nimport pandas as pd\nimport torch\n\nfrom case_study_2.models import configure_huggingface_cache, load_code_tokenizer, DEFAULT_NEOBERT_TOKENIZER\nfrom case_study_2.exp4.exp4_lora import train_lora_model_safe\nfrom utils import split_manifest\nfrom utils import evaluation\nfrom utils.evaluation import EvaluationConfig, select_f1_threshold\n\n\nEXP4_VERSION = "cs2-exp4-neobert-lora-v3-fixed-rank-rotating-5fold"\n\n\n@dataclass(frozen=True)\nclass Exp4Config:\n    """Declared reproducible configuration for CS2-EXP4 (NeoBERT LoRA fine-tuning)."""\n\n    experiment_name: str = "cs2_exp4_neobert_lora"\n\n    code_column: str = "normalized_code"\n    source_id_column: str = "source_row_id"\n    label_column: str = "label"\n    project_column: str = "project"\n    fold_column: str = "fold"\n\n    hf_cache_dir: Optional[str] = None\n    max_length: int = 512\n    train_batch_size: int = 16\n    grad_accum_steps: int = 2\n\n    # `epochs` is a ceiling, not a fixed count: training early-stops on\n    # validation PR-AUC and returns the best checkpoint.\n    epochs: int = 10\n    early_stopping: bool = True\n    patience: int = 2\n    min_epochs: int = 2\n\n    # Rank fixed at the literature-standard default for encoder models of\n    # this size (Hu et al. 2021 LoRA paper; HuggingFace PEFT docs), not\n    # searched, to keep NeoBERT fine-tuning within a practical compute\n    # budget (a 3-candidate x 3-inner-fold rank search would need 10x more\n    # NeoBERT trainings for a gain the literature says is usually marginal).\n    rank: int = 16\n    # Standard LoRA convention (alpha = 2 x rank), same reasoning as rank above.\n    lora_alpha_multiplier: int = 2\n    # 2e-4 is a standard LoRA fine-tuning learning rate; fixed for the same reason.\n    learning_rate: float = 2e-4\n\n    # Inner CV is still used, but only to calibrate the decision threshold\n    # on validation folds the outer test fold never touches.\n    inner_n_splits: int = 3\n    inner_random_state: int = 20260707\n\n    num_workers: int = 6\n    eval_batch_size: int = 64\n\n    n_splits: int = 5\n    random_state: int = 42\n    verbose: bool = True\n\n\ndef _checkpoint_paths(output_dir: Path, outer_fold_id: int) -> Dict[str, Path]:\n    """Filesystem locations for one outer fold\'s resumable checkpoint."""\n    root = output_dir / "checkpoints"\n    root.mkdir(parents=True, exist_ok=True)\n    prefix = f"outer_fold_{outer_fold_id}"\n    return {\n        "predictions": root / f"{prefix}_predictions.parquet",\n        "selected": root / f"{prefix}_selected.json",\n        "training": root / f"{prefix}_outer_training.json",\n        "history": root / f"{prefix}_training_history.parquet",\n    }\n\n\ndef _write_outer_checkpoint(output_dir: Path, outer_fold_id: int, predictions: pd.DataFrame, selected: dict, training: dict, history: pd.DataFrame) -> None:\n    """Persist one outer fold\'s final result so a later run can resume without refitting."""\n    paths = _checkpoint_paths(output_dir, outer_fold_id)\n    predictions.to_parquet(paths["predictions"], index=False)\n    with paths["selected"].open("w", encoding="utf-8") as f:\n        json.dump(selected, f, indent=2, default=str)\n    with paths["training"].open("w", encoding="utf-8") as f:\n        json.dump(training, f, indent=2, default=str)\n    history.to_parquet(paths["history"], index=False)\n\n\ndef _load_outer_checkpoint(output_dir: Path, outer_fold_id: int) -> Optional[dict]:\n    """Load one outer fold\'s checkpoint if it exists and is complete."""\n    paths = _checkpoint_paths(output_dir, outer_fold_id)\n    if not all(p.exists() for p in paths.values()):\n        return None\n    with paths["selected"].open("r", encoding="utf-8") as f:\n        selected = json.load(f)\n    with paths["training"].open("r", encoding="utf-8") as f:\n        training = json.load(f)\n    return {\n        "predictions": pd.read_parquet(paths["predictions"]), "selected": selected, "training": training,\n        "history": pd.read_parquet(paths["history"]),\n    }\n\n\ndef _update_run_state(state_path: Path, completed_folds, status: str) -> None:\n    """Persist which outer folds are done, for resumability and progress inspection."""\n    state = {\n        "status": status,\n        "updated_utc": datetime.now(timezone.utc).isoformat(),\n        "completed_outer_folds": sorted(int(f) for f in completed_folds),\n    }\n    with state_path.open("w", encoding="utf-8") as f:\n        json.dump(state, f, indent=2)\n\n\ndef _inner_manifest(frame: pd.DataFrame, config: Exp4Config, outer_fold_id: int) -> pd.DataFrame:\n    """Project-grouped, stratified 3-fold split of one outer fold\'s training rows."""\n    inner_split_config = split_manifest.SplitConfig(\n        n_splits=config.inner_n_splits,\n        random_state=config.inner_random_state + int(outer_fold_id),\n        source_id_column=config.source_id_column,\n        label_column=config.label_column,\n        group_column=config.project_column,\n    )\n    return split_manifest.create_project_grouped_manifest(\n        frame[[config.source_id_column, config.label_column, config.project_column]], config=inner_split_config\n    )\n\n\ndef _calibrate_threshold(\n    outer_train_df: pd.DataFrame,\n    tokenizer,\n    device: torch.device,\n    config: Exp4Config,\n    outer_fold_id: int,\n) -> dict:\n    """Calibrate the decision threshold via 3-fold inner CV at the fixed rank, on this fold\'s training projects only."""\n    lora_alpha = config.rank * config.lora_alpha_multiplier\n    inner_manifest_df = _inner_manifest(outer_train_df, config, outer_fold_id)\n\n    fold_predictions: List[pd.DataFrame] = []\n    for inner_fold_id in range(config.inner_n_splits):\n        tr_ids = set(inner_manifest_df.loc[inner_manifest_df["fold"] != inner_fold_id, config.source_id_column])\n        va_ids = set(inner_manifest_df.loc[inner_manifest_df["fold"] == inner_fold_id, config.source_id_column])\n        tr_frame = outer_train_df[outer_train_df[config.source_id_column].isin(tr_ids)].reset_index(drop=True)\n        va_frame = outer_train_df[outer_train_df[config.source_id_column].isin(va_ids)].reset_index(drop=True)\n\n        val_scores, tmp_model, _ = train_lora_model_safe(\n            tr_frame, va_frame, tokenizer, rank=config.rank, lora_alpha=lora_alpha,\n            learning_rate=config.learning_rate, epochs=config.epochs,\n            batch_size=config.train_batch_size, grad_accum_steps=config.grad_accum_steps,\n            eval_batch_size=config.eval_batch_size, num_workers=config.num_workers, device=device,\n            hf_cache_dir=config.hf_cache_dir, code_column=config.code_column, max_length=config.max_length,\n            log_prefix="    ", early_stopping=config.early_stopping, patience=config.patience,\n            min_epochs=config.min_epochs,\n        )\n        fold_predictions.append(\n            pd.DataFrame({"source_row_id": va_frame[config.source_id_column].values,\n                          "label": va_frame[config.label_column].astype(int).values, "y_score": val_scores})\n        )\n        del tmp_model\n        gc.collect()\n        torch.cuda.empty_cache()\n\n    pooled = pd.concat(fold_predictions, ignore_index=True)\n    selected_threshold, threshold_metrics = select_f1_threshold(pooled["label"], pooled["y_score"])\n\n    print(f"[nested] rank={config.rank} (alpha={lora_alpha}, fixed) | threshold={selected_threshold:.2f} "\n          f"(inner validation F1={threshold_metrics[\'f1\']:.4f})")\n\n    return {\n        "rank": config.rank,\n        "lora_alpha": lora_alpha,\n        "decision_threshold": selected_threshold,\n        "inner_validation_f1": float(threshold_metrics["f1"]),\n    }\n\n\ndef run_exp4_nested_rank(\n    dataset_frame: pd.DataFrame,\n    manifest: pd.DataFrame,\n    config: Exp4Config,\n    output_dir: Path,\n    resume: bool = True,\n    additional_metadata: Optional[Dict[str, Any]] = None,\n) -> Dict[str, Any]:\n    """Run the official rotating 5-fold CS2-EXP4 experiment, with a 3-fold inner-CV threshold calibration per outer fold."""\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    if device.type != "cuda":\n        raise RuntimeError("EXP-4 LoRA fine-tuning requires a CUDA device.")\n\n    output_dir = Path(output_dir)\n    output_dir.mkdir(parents=True, exist_ok=True)\n    state_path = output_dir / "exp4_nested_run_state.json"\n\n    configure_huggingface_cache(config.hf_cache_dir)\n    tokenizer = load_code_tokenizer(DEFAULT_NEOBERT_TOKENIZER, hf_cache_dir=config.hf_cache_dir)\n\n    fold_ids = sorted(manifest[config.fold_column].unique().tolist())\n    print(f"[nested] Starting EXP-4 rotating {len(fold_ids)}-fold run (NeoBERT LoRA), rank={config.rank} (fixed)")\n    print(f"[nested] Threshold calibration: {config.inner_n_splits}-fold inner CV | "\n          f"Refit phase: {config.epochs} epoch(s) ceiling, full outer-train")\n\n    oof_parts = []\n    selected_rows = []\n    outer_training_rows = []\n    training_histories = []\n    completed_folds = []\n    t0 = time.time()\n\n    for outer_fold_id in fold_ids:\n        checkpoint = _load_outer_checkpoint(output_dir, outer_fold_id) if resume else None\n        if checkpoint is not None:\n            oof_parts.append(checkpoint["predictions"])\n            selected_rows.append(checkpoint["selected"])\n            outer_training_rows.append(checkpoint["training"])\n            training_histories.append(checkpoint["history"])\n            completed_folds.append(outer_fold_id)\n            if config.verbose:\n                print(f"[nested] Outer fold {outer_fold_id}: loaded from checkpoint, skipping.")\n            continue\n\n        fold_t0 = time.time()\n        print(f"\\n=================== OUTER FOLD {outer_fold_id} ({len(completed_folds)+1}/{len(fold_ids)}) ===================")\n\n        outer_train_ids = manifest.loc[manifest[config.fold_column] != outer_fold_id, config.source_id_column]\n        outer_val_ids = manifest.loc[manifest[config.fold_column] == outer_fold_id, config.source_id_column]\n        outer_train_df = dataset_frame[dataset_frame[config.source_id_column].isin(outer_train_ids)].reset_index(drop=True)\n        outer_val_df = dataset_frame[dataset_frame[config.source_id_column].isin(outer_val_ids)].reset_index(drop=True)\n        print(f"[nested] outer_train={len(outer_train_df)} rows | outer_val={len(outer_val_df)} rows")\n\n        selection = _calibrate_threshold(outer_train_df, tokenizer, device, config, outer_fold_id)\n\n        print(f"[nested] --- final refit on full outer_train, rank={selection[\'rank\']}, "\n              f"alpha={selection[\'lora_alpha\']}, {config.epochs} epochs ---")\n        refit_t0 = time.time()\n        outer_val_scores, final_outer_model, refit_history = train_lora_model_safe(\n            outer_train_df, outer_val_df, tokenizer, rank=selection["rank"],\n            lora_alpha=selection["lora_alpha"], learning_rate=config.learning_rate,\n            epochs=config.epochs, batch_size=config.train_batch_size, grad_accum_steps=config.grad_accum_steps,\n            eval_batch_size=config.eval_batch_size, num_workers=config.num_workers, device=device,\n            hf_cache_dir=config.hf_cache_dir, code_column=config.code_column, max_length=config.max_length,\n            log_prefix="    ", early_stopping=config.early_stopping, patience=config.patience,\n            min_epochs=config.min_epochs,\n        )\n        fold_history = pd.DataFrame(refit_history)\n        fold_history.insert(0, "fold", int(outer_fold_id))\n        print(f"[nested] refit done in {(time.time()-refit_t0)/60:.1f} min")\n\n        fold_oof = pd.DataFrame({\n            config.source_id_column: outer_val_df[config.source_id_column].values,\n            config.project_column: outer_val_df[config.project_column].values,\n            "label": outer_val_df[config.label_column].astype(int).values,\n            "y_score": outer_val_scores,\n            "fold": outer_fold_id,\n        })\n\n        selected_row = {"outer_fold_id": outer_fold_id, **selection}\n        training_row = {\n            **selected_row,\n            "n_train": int(len(outer_train_df)),\n            "n_val": int(len(outer_val_df)),\n            "elapsed_minutes": (time.time() - fold_t0) / 60,\n        }\n\n        if outer_fold_id == fold_ids[-1]:\n            final_outer_model.save_pretrained(output_dir / "final_exp4_lora_adapter")\n            print(f"[nested] saved final fold LoRA adapter to {output_dir / \'final_exp4_lora_adapter\'}")\n\n        _write_outer_checkpoint(output_dir, outer_fold_id, fold_oof, selected_row, training_row, fold_history)\n\n        oof_parts.append(fold_oof)\n        selected_rows.append(selected_row)\n        outer_training_rows.append(training_row)\n        training_histories.append(fold_history)\n        completed_folds.append(outer_fold_id)\n\n        _update_run_state(state_path, completed_folds, status="running")\n\n        del outer_train_df, outer_val_df, final_outer_model\n        gc.collect()\n        torch.cuda.empty_cache()\n\n        print(f"[nested] Outer fold {outer_fold_id} done in {training_row[\'elapsed_minutes\']:.1f} min | "\n              f"total elapsed {(time.time()-t0)/60:.1f} min")\n\n    oof_predictions = pd.concat(oof_parts, axis=0).reset_index(drop=True)\n    selected_df = pd.DataFrame(selected_rows)\n    outer_training_df = pd.DataFrame(outer_training_rows)\n    training_history_df = pd.concat(training_histories, ignore_index=True).sort_values(["fold", "epoch"]).reset_index(drop=True)\n\n    mean_threshold = float(selected_df["decision_threshold"].mean())\n    eval_config = EvaluationConfig(threshold=mean_threshold, expected_n_folds=len(fold_ids))\n    eval_results = evaluation.evaluate_oof_predictions(oof_predictions, config=eval_config)\n\n    artifacts = {\n        "oof_predictions": output_dir / "exp4_nested_oof_predictions.parquet",\n        "selected_per_fold": output_dir / "exp4_selected_per_outer_fold.csv",\n        "outer_training_audit": output_dir / "exp4_outer_training_audit.csv",\n        "training_history": output_dir / "exp4_training_history.csv",\n        "run_metadata": output_dir / "exp4_nested_run_metadata.json",\n    }\n    eval_results["predictions"].to_parquet(artifacts["oof_predictions"], index=False)\n    selected_df.to_csv(artifacts["selected_per_fold"], index=False)\n    outer_training_df.to_csv(artifacts["outer_training_audit"], index=False)\n    training_history_df.to_csv(artifacts["training_history"], index=False)\n\n    metadata = {\n        "exp4_version": EXP4_VERSION,\n        "config": asdict(config),\n        "runtime_seconds": time.time() - t0,\n        **(additional_metadata or {}),\n    }\n    with open(artifacts["run_metadata"], "w", encoding="utf-8") as f:\n        json.dump(metadata, f, indent=2, default=str)\n\n    _update_run_state(state_path, completed_folds, status="completed")\n    print(f"\\n[nested] EXP-4 rotating {len(fold_ids)}-fold run complete in {(time.time()-t0)/60:.1f} min")\n\n    return {\n        "oof_predictions": eval_results["predictions"],\n        "evaluation": eval_results,\n        "selected": selected_df,\n        "outer_fold_training": outer_training_df,\n        "training_history": training_history_df,\n        "artifacts": artifacts,\n        "tokenizer": tokenizer,\n    }\n\n\n__all__ = [\n    "EXP4_VERSION",\n    "Exp4Config",\n    "run_exp4_nested_rank",\n]\n')
print("Wrote", "case_study_2/exp4/exp4_nested_rank.py")

## 6. Import project modules


In [ ]:
import sys

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

for mod_name in list(sys.modules.keys()):
    if mod_name.startswith("case_study_2") or mod_name.startswith("case_study_1") or mod_name.startswith("utils"):
        del sys.modules[mod_name]

from case_study_2.models import (
    configure_huggingface_cache,
    load_code_tokenizer,
    DEFAULT_NEOBERT_MODEL,
    DEFAULT_NEOBERT_TOKENIZER,
    count_trainable_parameters,
    CodeSequenceClassifier,
    infer_lora_target_modules,
)
from case_study_2.exp4.exp4_lora import train_lora_model_safe
from case_study_2.exp4.exp4_nested_rank import Exp4Config, run_exp4_nested_rank
from utils import split_manifest
from utils.confidence_intervals import bootstrap_metric_ci, format_ci_report


## 7. Load the downsampled dataset and shared 5-fold manifest

In [ ]:
import pandas as pd

if not DOWNSAMPLED_PARQUET.is_file():
    raise FileNotFoundError(f"Missing downsampled parquet: {DOWNSAMPLED_PARQUET}")
if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(f"Missing manifest: {MANIFEST_PATH}")

full_df = pd.read_parquet(DOWNSAMPLED_PARQUET)
manifest_df = split_manifest.load_manifest(MANIFEST_PATH, config=split_manifest.SplitConfig(n_splits=5, random_state=42))

print("full_df rows:", len(full_df))
print("manifest_df rows:", len(manifest_df))

fold_summary_diag = split_manifest.summarize_manifest(
    manifest_df, config=split_manifest.SplitConfig(n_splits=5, random_state=42)
)
display(fold_summary_diag)

fold_size_ratio = fold_summary_diag["test_rows"].max() / fold_summary_diag["test_rows"].min()
print(f"Fold test-size balance: smallest={fold_summary_diag['test_rows'].min()} rows, "
      f"largest={fold_summary_diag['test_rows'].max()} rows, ratio={fold_size_ratio:.2f}x")
if fold_size_ratio > 2.0:
    print("WARNING: fold sizes are notably imbalanced (ratio > 2x) -- "
          "regenerate the manifest via scope2_preprocessing.ipynb with an updated "
          "DOWNSAMPLE_MAX_ROWS_PER_PROJECT.")
else:
    print("Fold sizes look reasonably balanced.")


## 8. Build the dataset frame

In [ ]:
required_columns = {"source_row_id", "normalized_code", "abstracted_code_v1", "label", "project"}
missing_columns = required_columns - set(full_df.columns)
if missing_columns:
    raise ValueError(f"Missing columns in full_df: {missing_columns}")

full_indexed = full_df.set_index("source_row_id", drop=False)
manifest_ids = set(manifest_df["source_row_id"].tolist())
if manifest_ids != set(full_indexed.index):
    raise RuntimeError(
        "Manifest coverage does not match the downsampled dataset exactly. "
        f"Missing={len(set(full_indexed.index) - manifest_ids)}, extra={len(manifest_ids - set(full_indexed.index))}"
    )

dataset_frame = full_indexed.loc[list(manifest_ids)].reset_index(drop=True)
print("dataset_frame rows:", len(dataset_frame))
print("Unique projects:", dataset_frame["project"].nunique())


## 9. Configuration object


In [ ]:
exp4_config = Exp4Config(
    hf_cache_dir=HF_CACHE_DIR,
    code_column=CODE_COLUMN,
    rank=RANK,
    epochs=EPOCHS,
    train_batch_size=TRAIN_BATCH_SIZE,
    grad_accum_steps=GRAD_ACCUM_STEPS,
    num_workers=NUM_WORKERS,
    eval_batch_size=EVAL_BATCH_SIZE,
)

print("Code column:", exp4_config.code_column)
print("rank (fixed):", exp4_config.rank)
print("lora_alpha_multiplier:", exp4_config.lora_alpha_multiplier)
print("learning_rate:", exp4_config.learning_rate)
print("inner_n_splits:", exp4_config.inner_n_splits)


## 10. Sanity-check LoRA target modules


In [16]:
configure_huggingface_cache(HF_CACHE_DIR)
_tok_check = load_code_tokenizer(DEFAULT_NEOBERT_TOKENIZER, hf_cache_dir=HF_CACHE_DIR)

_probe_model = CodeSequenceClassifier(model_name=DEFAULT_NEOBERT_MODEL, freeze_backbone=False, dtype_policy="bfloat16", hf_cache_dir=HF_CACHE_DIR)
_target_modules = infer_lora_target_modules(_probe_model)
print("Inferred LoRA target_modules:", _target_modules)

del _probe_model
torch.cuda.empty_cache()


2026-08-31 18:48:55.003011: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-31 18:48:55.017327: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1788202135.033261   20022 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1788202135.038012   20022 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-08-31 18:48:55.055574: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

Inferred LoRA target_modules: ['qkv']


## 11. Smoke test on a small subsample


In [ ]:
import time
from sklearn.metrics import average_precision_score

if RUN_SMOKE_TEST:
    sample_df = dataset_frame.sample(n=min(2000, len(dataset_frame)), random_state=42).reset_index(drop=True)
    smoke_train = sample_df.iloc[:1500].reset_index(drop=True)
    smoke_val = sample_df.iloc[1500:].reset_index(drop=True)

    print(f"[smoke] train={len(smoke_train)} rows | val={len(smoke_val)} rows")

    t0 = time.time()
    smoke_scores, smoke_model, _ = train_lora_model_safe(
        smoke_train, smoke_val, _tok_check, rank=exp4_config.rank,
        lora_alpha=exp4_config.rank * exp4_config.lora_alpha_multiplier,
        learning_rate=exp4_config.learning_rate, epochs=1,
        batch_size=exp4_config.train_batch_size, grad_accum_steps=exp4_config.grad_accum_steps,
        eval_batch_size=exp4_config.eval_batch_size, num_workers=exp4_config.num_workers,
        device=DEVICE, hf_cache_dir=HF_CACHE_DIR, code_column=exp4_config.code_column,
        max_length=exp4_config.max_length,
    )
    smoke_prauc = float(average_precision_score(smoke_val[exp4_config.label_column].values, smoke_scores))
    print(f"[smoke] PR-AUC={smoke_prauc:.4f} | elapsed={(time.time()-t0)/60:.1f} min")

    del smoke_model
    torch.cuda.empty_cache()
else:
    print("RUN_SMOKE_TEST=False; skipping.")


## 12. Official rotating 5-fold run

In [ ]:
if RUN_OFFICIAL:
    results = run_exp4_nested_rank(
        dataset_frame=dataset_frame,
        manifest=manifest_df,
        config=exp4_config,
        output_dir=EXP4_OUTPUT_DIR,
        additional_metadata={
            "input_parquet": str(DOWNSAMPLED_PARQUET),
            "manifest_path": str(MANIFEST_PATH),
        },
    )
    print("Pooled OOF metrics (secondary cross-check):")
    display(pd.DataFrame([{"metric": k, "value": v} for k, v in results["evaluation"]["pooled_metrics"].items()]))
    print("Mean +/- std across the 5 outer folds (headline result):")
    display(results["evaluation"]["fold_summary"])
    print("Selected hyperparameters by outer fold:")
    display(results["selected"])
else:
    results = None
    print("RUN_OFFICIAL=False; skipping.")


## 13. Confidence interval on pooled OOF PR-AUC (ad hoc)

In [ ]:
exp4_oof_ci = bootstrap_metric_ci(
    results["oof_predictions"],
    metric="average_precision_pr_auc",
    n_bootstrap=1000,
    random_state=42,
)
print(format_ci_report(exp4_oof_ci))


## 14. Cleanup

In [22]:
import gc
import shutil
import torch

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("VRAM allocated:", torch.cuda.memory_allocated() / 1e9, "GB")

total, used, free = shutil.disk_usage(WORKSPACE_ROOT)
print(f"Disk usage at {WORKSPACE_ROOT}: {used/1e9:.1f} GB used / {total/1e9:.1f} GB total ({free/1e9:.1f} GB free)")
if used / 1e9 > STORAGE_CAP_GB:
    print(f"WARNING: workspace usage exceeds the {STORAGE_CAP_GB} GB storage cap -- consider pruning old checkpoints under {EXP4_OUTPUT_DIR}.")


VRAM allocated: 0.01703936 GB
Disk usage at /workspace: 5007.2 GB used / 5714.2 GB total (418.9 GB free)
